In [0]:
from pyspark.sql import SparkSession
from src.analysis.new_customers import calculate_new_customers
spark = SparkSession.getActiveSession()
result = calculate_new_customers(spark)
display(result)

In [0]:
from pyspark.sql import functions as F

orders = spark.table("workspace.marketing_silver.orders")
first_orders = (
    orders
    .groupBy("o_custkey")
    .agg(F.min("o_orderdate").alias("first_order_date"))
)
q1_customers = (
    first_orders
    .filter(
        (F.col("first_order_date") >= "1996-01-01")
        & (F.col("first_order_date") < "1996-04-01")
    )
    .orderBy("first_order_date")
)
display(q1_customers)

In [0]:
monthly_new_customers = (
    q1_customers
    .withColumn(
        "month",
        F.date_format("first_order_date", "yyyy-MM")
    )
    .groupBy("month")
    .agg(
        F.count("*").alias("new_customers")
    )
    .orderBy("month")
)
display(monthly_new_customers)

Databricks visualization. Run in Databricks to view.